<a href="https://colab.research.google.com/github/NourhanFarag/nourhan-flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NourhanFarag/nourhan-flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.950
Hand rule  Precision@50: 0.660


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.950   vs   tree 0.650
Precision@50:  hand rule 0.660   vs   tree 0.640


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# Experiment 1: Compare shallow decision trees with different maximum depths.
# Question: Does a deeper tree improve Precision@50, and how much complexity does it add?

depth_results = []
depth_models = {}

for max_depth in [2, 3, 4]:
    # Train a decision tree using the notebook's original feature set.
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X, y)

    # Save each trained model so we can inspect its rules later.
    depth_models[max_depth] = model

    # Rank pages using the probability that each page is declining.
    decline_scores = model.predict_proba(X)[:, 1]

    # Identify the feature selected for the tree's first decision.
    root_feature_index = model.tree_.feature[0]
    first_split_feature = features[root_feature_index]

    # Store both ranking performance and model complexity.
    depth_results.append({
        "requested_max_depth": max_depth,
        "actual_depth": model.get_depth(),
        "leaf_count": model.get_n_leaves(),
        "first_split": first_split_feature,
        "precision_at_20": precision_at_k(decline_scores, y, 20),
        "precision_at_50": precision_at_k(decline_scores, y, 50)
    })

# Convert the collected results into a readable comparison table.
depth_results_df = pd.DataFrame(depth_results)

# Round only the metric columns for clearer display.
depth_results_df[["precision_at_20", "precision_at_50"]] = (
    depth_results_df[["precision_at_20", "precision_at_50"]]
    .round(3)
)

depth_results_df

,requested_max_depth,actual_depth,leaf_count,first_split,precision_at_20,precision_at_50
0,2,2,4,impressions_90d,0.65,0.64
1,3,3,8,impressions_90d,0.65,0.66
2,4,4,16,impressions_90d,0.70,0.72


In [8]:
# Inspect the learned rules to judge how readability changes with depth.

for depth in [3, 4]:
    print("=" * 70)
    print(
        f"Decision tree with max_depth={depth} "
        f"({depth_models[depth].get_n_leaves()} leaves)"
    )
    print("=" * 70)

    print(
        export_text(
            depth_models[depth],
            feature_names=features
        )
    )

Decision tree with max_depth=3 (8 leaves)
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0

Decision tree with max_depth=4 (16 leaves)
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |-

In [9]:
# Experiment 2: Change the feature set.
# Question: When impressions are removed, which feature does the tree
# choose for its first split, and how does ranking performance change?

modified_features = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

# Prepare the modified feature table safely.
modified_X = (
    df[modified_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Use depth 3 so the tree remains reasonably readable.
modified_tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

modified_tree.fit(modified_X, y)

# Rank pages by their estimated probability of being declining.
modified_scores = modified_tree.predict_proba(modified_X)[:, 1]

# Identify the feature selected for the first decision.
root_feature_index = modified_tree.tree_.feature[0]
modified_first_split = modified_features[root_feature_index]

print("Modified feature set:", modified_features)
print("First split selected:", modified_first_split)
print(
    "Precision@20:",
    round(precision_at_k(modified_scores, y, 20), 3)
)
print(
    "Precision@50:",
    round(precision_at_k(modified_scores, y, 50), 3)
)

print("\nReadable rules:")
print(
    export_text(
        modified_tree,
        feature_names=modified_features
    )
)

Modified feature set: ['content_age_days', 'days_since_last_update', 'avg_position', 'ctr', 'word_count', 'engagement_rate']
First split selected: avg_position
Precision@20: 0.95
Precision@50: 0.78

Readable rules:
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- class: 0
|   |   |--- word_count >  669.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- avg_position <= 38.45
|   |   |   |--- class: 0
|   |   |--- avg_position >  38.45
|   |   |   |--- class: 0



In [10]:
# Experiment 3: Evaluate the hand rule and the original depth-2 tree
# using a client-holdout split.
#
# Question: Does the tree still perform well when tested on clients
# that were completely absent from training?

from sklearn.model_selection import GroupShuffleSplit

# Convert the target to a NumPy array so indexing is consistent.
y_array = np.asarray(y)

# Use client_id as the grouping variable.
# Every client's pages must belong entirely to either training or testing.
client_groups = df["client_id"].astype(str)

# Create one reproducible split:
# approximately 80% of clients for training and 20% for testing.
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    group_split.split(
        X,
        y_array,
        groups=client_groups
    )
)

# Separate the original feature table.
# X is a Pandas DataFrame, so .iloc is appropriate here.
X_train = X.iloc[train_index]
X_test = X.iloc[test_index]

# y_array is a NumPy array, so use bracket indexing instead of .iloc.
y_train = y_array[train_index]
y_test = y_array[test_index]

# Train the original readable depth-2 tree only on training clients.
holdout_tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

holdout_tree.fit(X_train, y_train)

# Score only pages belonging to unseen test clients.
holdout_tree_scores = holdout_tree.predict_proba(X_test)[:, 1]

# The hand-written rule does not require training.
# Evaluate it on exactly the same held-out test rows.
holdout_rule_scores = (
    df.iloc[test_index]["hand_rule_score"]
    .to_numpy()
)

# Confirm that no client appears in both sets.
train_clients = set(client_groups.iloc[train_index])
test_clients = set(client_groups.iloc[test_index])
client_overlap = train_clients.intersection(test_clients)

print("Training rows:", len(train_index))
print("Testing rows:", len(test_index))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Clients appearing in both sets:", len(client_overlap))

print("\nClient-holdout Precision@20:")
print(
    "Hand rule:",
    round(precision_at_k(holdout_rule_scores, y_test, 20), 3)
)
print(
    "Depth-2 tree:",
    round(precision_at_k(holdout_tree_scores, y_test, 20), 3)
)

print("\nClient-holdout Precision@50:")
print(
    "Hand rule:",
    round(precision_at_k(holdout_rule_scores, y_test, 50), 3)
)
print(
    "Depth-2 tree:",
    round(precision_at_k(holdout_tree_scores, y_test, 50), 3)
)

print("\nTree learned from training clients:")
print(
    export_text(
        holdout_tree,
        feature_names=features
    )
)

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Clients appearing in both sets: 0

Client-holdout Precision@20:
Hand rule: 0.4
Depth-2 tree: 0.7

Client-holdout Precision@50:
Hand rule: 0.56
Depth-2 tree: 0.72

Tree learned from training clients:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



In [11]:
# Experiment 4: Compare tree depth and feature sets using the same
# client-holdout split created in Experiment 3.
#
# Question:
# Which combination gives the strongest Precision@50 on unseen clients
# while remaining reasonably interpretable?

# Define the two feature configurations.
feature_sets = {
    "original": {
        "columns": features,
        "data": X
    },
    "modified": {
        "columns": modified_features,
        "data": modified_X
    }
}

combined_results = []
combined_models = {}

# Train depths 2, 3, and 4 for both feature sets.
for feature_set_name, feature_info in feature_sets.items():
    feature_columns = feature_info["columns"]
    feature_data = feature_info["data"]

    # Use the exact same client-holdout rows for every model,
    # making the comparisons consistent.
    current_X_train = feature_data.iloc[train_index]
    current_X_test = feature_data.iloc[test_index]

    for max_depth in [2, 3, 4]:
        model = DecisionTreeClassifier(
            max_depth=max_depth,
            class_weight="balanced",
            random_state=42
        )

        model.fit(current_X_train, y_train)

        # Rank only pages from unseen test clients.
        test_scores = model.predict_proba(current_X_test)[:, 1]

        # Identify the feature used in the first split.
        root_feature_index = model.tree_.feature[0]
        first_split_feature = feature_columns[root_feature_index]

        model_name = f"{feature_set_name}_depth_{max_depth}"
        combined_models[model_name] = model

        combined_results.append({
            "feature_set": feature_set_name,
            "requested_depth": max_depth,
            "actual_depth": model.get_depth(),
            "leaf_count": model.get_n_leaves(),
            "first_split": first_split_feature,
            "precision_at_20": precision_at_k(
                test_scores,
                y_test,
                20
            ),
            "precision_at_50": precision_at_k(
                test_scores,
                y_test,
                50
            )
        })

# Convert the collected results into a readable table.
combined_results_df = pd.DataFrame(combined_results)

# Round the evaluation metrics for display.
combined_results_df[["precision_at_20", "precision_at_50"]] = (
    combined_results_df[
        ["precision_at_20", "precision_at_50"]
    ].round(3)
)

# Rank primarily by Precision@50, then by fewer leaves
# when performance is tied.
combined_results_df = combined_results_df.sort_values(
    ["precision_at_50", "leaf_count"],
    ascending=[False, True]
).reset_index(drop=True)

combined_results_df

,feature_set,requested_depth,actual_depth,leaf_count,first_split,precision_at_20,precision_at_50
0,original,2,2,4,impressions_90d,0.70,0.72
1,original,3,3,8,impressions_90d,0.70,0.66
2,original,4,4,16,impressions_90d,0.55,0.62
3,modified,2,2,4,avg_position,0.55,0.60
4,modified,4,4,16,avg_position,0.70,0.60
5,modified,3,3,8,avg_position,0.50,0.56


### Your turn: complexity, features, and client-holdout validation

**Research question:**  
Does increasing decision-tree depth or changing the feature set improve Precision@50, and do those improvements generalize to clients that were not present during training?

**Method:**  
I first compared decision trees with maximum depths of 2, 3, and 4 using the original features and in-sample scoring. I then removed `impressions_90d`, added `engagement_rate`, and inspected which feature the modified tree selected for its first split. Finally, I created a client-holdout split, ensuring that no client appeared in both training and testing, and compared both feature sets at all three depths on the same held-out clients.

**Observed in-sample results:**  
Using the original features, Precision@50 increased from 0.64 at depth 2 to 0.72 at depth 4, while the number of leaves increased from 4 to 16. The modified depth-3 tree achieved an in-sample Precision@50 of 0.78 and selected `avg_position` as its first split. Although `engagement_rate` was available, the tree did not select it for any displayed split.

**Observed client-holdout results:**  
The original depth-2 tree achieved the strongest holdout Precision@50 of 0.72. Increasing the original tree depth reduced holdout Precision@50 to 0.66 at depth 3 and 0.62 at depth 4. The modified feature-set models achieved lower holdout Precision@50 values between 0.56 and 0.60. No client appeared in both the training and testing groups.

**Interpretation:**  
For this client-holdout split, greater tree complexity improved in-sample performance but did not generalize better to unseen clients. The original depth-2 tree provided the strongest holdout ranking while also being the easiest model to read. This illustrates that the model with the highest training-data result is not necessarily the best model for new data.

**Limitations:**  
This experiment used one reproducible client-holdout split, so the results should be treated as observed and directional. Repeating the evaluation across several client-based splits would provide a more stable estimate of model performance.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.